# Datathon 2026 - V5 Hybrid Upgrade

B?n notebook n?y gi? nguy?n logic d? b?o c?a V5 Hybrid, ch? s?p l?i ph?n tr?nh b?y ?? l?m r? c?c y?u c?u ch?m ?i?m:

- **Pipeline r? r?ng**: d? li?u -> feature known-future/past-only -> walk-forward CV -> model library -> blend/calibration -> COGS reconciliation -> submission/diagnostics.
- **Cross-validation ??ng chi?u th?i gian**: `make_splits` t?o expanding-window folds; trong `train_target_hybrid`, m?i fold lu?n train tr?n c?c ng?y tr??c validation.
- **Explainability b?ng SHAP ho?c t??ng ???ng**: notebook gi?i th?ch theo t?ng m? h?nh b?ng OOF metrics, `global_weights`, `bucket_weights`, calibration, Kalman gate v? `final_components`; ??y l? ph?n t??ng ???ng model attribution, kh?ng can thi?p v?o d? b?o.
- **Tu?n th? r?ng bu?c**: test ch? d?ng `Date` v? feature bi?t tr??c; kh?ng d?ng `Revenue/COGS` t??ng lai l?m feature; anchor submission ch? d?ng ?? t?o probe blend, kh?ng d?ng l?m label train.

> Ghi ch? b?o to?n k?t qu?: c?c code cell b?n d??i ???c gi? nguy?n, kh?ng ??i logic model, tham s?, th? t? train hay c?ch ghi submission.


## Pipeline Map

Notebook c? m?t code cell l?n ?? gi? m?i h?m trong c?ng m?i tr??ng ch?y. Lu?ng th?c t? trong code:

| B??c | H?m/kh?i code | Vai tr? |
|---|---|---|
| 1 | `load_data` | ??c `sales.csv`, `sample_submission.csv`, v? `promotions.csv` n?u c?. |
| 2 | `build_feature_frame` | T?o calendar, T?t, mega-sale, promotion v? interaction features bi?t tr??c. |
| 3 | `make_lag_features`, `lag_row_from_history` | T?o lag/rolling t? l?ch s? qu? kh?; validation/test recursive ch? c?p nh?t b?ng l?ch s? ho?c prediction tr??c ??. |
| 4 | `make_splits` | T?o expanding-window folds theo ??ng th? t? th?i gian. |
| 5 | `train_target_hybrid` | Train t?ng target qua seasonal anchor + XGBoost candidates + blend/calibration/Kalman gate. |
| 6 | `run_pipeline` | Ch?y `Revenue`, `COGS_direct`, `COGS_ratio`, reconcile COGS r?i ghi submission v? diagnostics. |

C?c feature d?ng cho test l? deterministic/known-future (`Date`, calendar, event, promotion plan) ho?c ???c t?o t? l?ch s?/prediction qu? kh? trong nh?nh lag. V? v?y code kh?ng ??c `Revenue` ho?c `COGS` c?a test l?m input.


In [1]:
# Datathon 2026 Round 1 - V5 Hybrid Upgrade
# Combines the strongest ideas from:
#   1) v3_optimize: compact known-future XGBoost model library + MAE blend/calibration
#   2) v4_kalman_walkforward: richer calendar/Tet/promo features, seasonal anchor,
#      optional recursive lag candidate, horizon-bucket blend, and safe Kalman residual gate.
#
# Outputs:
#   - submission_v5_hybrid_model_only.csv
#   - submission_v5_recommended.csv
#   - optional conservative blends with an anchor/best submission if available
#   - diagnostics CSV/JSON for report/reproducibility

from __future__ import annotations

import json
import math
import os
import re
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

SEED = 42
DATE_COL = "date"
TARGETS = ["Revenue", "COGS"]

TET_DATES = pd.to_datetime([
    "2012-01-23", "2013-02-10", "2014-01-31", "2015-02-19", "2016-02-08",
    "2017-01-28", "2018-02-16", "2019-02-05", "2020-01-25", "2021-02-12",
    "2022-02-01", "2023-01-22", "2024-02-10", "2025-01-29", "2026-02-17",
])

MEGA_DAYS = {
    "dd_3_3": (3, 3),
    "valentine": (2, 14),
    "womens_day": (3, 8),
    "dd_6_6": (6, 6),
    "dd_9_9": (9, 9),
    "dd_10_10": (10, 10),
    "dd_11_11": (11, 11),
    "dd_12_12": (12, 12),
    "christmas": (12, 25),
}

LAGS = [1, 2, 3, 7, 14, 21, 28, 35, 56, 91, 182, 364, 365, 366, 728, 730]
ROLL_WINDOWS = [7, 14, 28, 56, 91, 182, 365]


@dataclass
class V5Config:
    seed: int = SEED
    val_days: int = 548
    step_days: int = 365
    max_folds: int = 3
    n_blend_samples: int = 12000
    use_extended_recurring_promos: bool = True
    use_recursive_lag_candidate: bool = True
    use_horizon_bucket_blend: bool = True
    use_kalman_gate: bool = True
    train_recent_years_default: Optional[int] = None
    train_recent_years_strong: int = 4
    n_jobs: int = 2
    output_dir: str = "/kaggle/working"
    data_dir: Optional[str] = None
    anchor_submission_path: Optional[str] = None
    # Very conservative leaderboard probing. The anchor is the user's current best submission,
    # not used as a label; it is only used to produce candidate blends.
    anchor_blend_weights: Tuple[float, ...] = (0.05, 0.10, 0.15, 0.20, 0.30)
    recommended_anchor_weight: float = 0.10


CFG = V5Config()


def set_seed(seed: int = SEED) -> None:
    np.random.seed(seed)


def detect_xgb_device() -> str:
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
    except Exception:
        pass
    return "cpu"


XGB_DEVICE = detect_xgb_device()


def has_competition_files(folder: Path) -> bool:
    if not folder.exists() or not folder.is_dir():
        return False
    names = {p.name.lower() for p in folder.glob("*") if p.is_file()}
    return {"sales.csv", "sample_submission.csv"}.issubset(names)


def find_data_dir(explicit: Optional[str | Path] = None) -> Path:
    candidates: List[Path] = []
    if explicit is not None:
        candidates.append(Path(explicit))
    candidates += [
        Path("/kaggle/input/competitions/datathon-2026-round-1"),
        Path("/kaggle/input/datathon-2026-round-1"),
        Path("/kaggle/input/datathon-2026-round-1/dataset"),
        Path("/kaggle/input/datasets/luquang231/datathonvinu"),
        Path("/mnt/data/datathon_extracted"),
        Path.cwd(),
        Path.cwd() / "dataset",
        Path.cwd().parent / "dataset",
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for root, dirs, files in os.walk(kaggle_input):
            root_path = Path(root)
            candidates.append(root_path)
            if len(candidates) > 500:
                break
    seen = set()
    for folder in candidates:
        try:
            key = str(folder.resolve())
        except Exception:
            key = str(folder)
        if key in seen:
            continue
        seen.add(key)
        if has_competition_files(folder):
            return folder
    raise FileNotFoundError("Cannot find sales.csv and sample_submission.csv. Set cfg.data_dir explicitly.")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    return out


def load_data(cfg: V5Config):
    data_dir = find_data_dir(cfg.data_dir)
    sales = pd.read_csv(data_dir / "sales.csv")
    sample = pd.read_csv(data_dir / "sample_submission.csv")
    sales = normalize_columns(sales)
    sample = normalize_columns(sample)
    sales["Date"] = pd.to_datetime(sales["Date"], errors="coerce")
    sample["Date"] = pd.to_datetime(sample["Date"], errors="coerce")
    sales = sales.dropna(subset=["Date"]).drop_duplicates("Date").sort_values("Date").reset_index(drop=True)
    sample = sample.dropna(subset=["Date"]).reset_index(drop=True)
    for c in ["Revenue", "COGS"]:
        sales[c] = pd.to_numeric(sales[c], errors="coerce").interpolate().ffill().bfill().clip(lower=0)

    promos = None
    promo_path = data_dir / "promotions.csv"
    if promo_path.exists():
        promos = pd.read_csv(promo_path)
        promos = normalize_columns(promos)
        for c in ["start_date", "end_date"]:
            if c in promos.columns:
                promos[c] = pd.to_datetime(promos[c], errors="coerce")
    return data_dir, sales, sample[["Date"]].copy(), promos


def black_friday(year: int) -> pd.Timestamp:
    nov = pd.date_range(f"{year}-11-01", f"{year}-11-30", freq="D")
    return nov[nov.dayofweek == 4][3]


def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    d = out[DATE_COL].dt
    out["day_of_week"] = d.dayofweek.astype(int)
    out["day_of_month"] = d.day.astype(int)
    out["day_of_year"] = d.dayofyear.astype(int)
    out["week_of_year"] = d.isocalendar().week.astype(int)
    out["month"] = d.month.astype(int)
    out["quarter"] = d.quarter.astype(int)
    out["year"] = d.year.astype(int)
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["is_month_start"] = d.is_month_start.astype(int)
    out["is_month_end"] = d.is_month_end.astype(int)
    out["is_quarter_start"] = d.is_quarter_start.astype(int)
    out["is_quarter_end"] = d.is_quarter_end.astype(int)
    out["is_year_start"] = d.is_year_start.astype(int)
    out["is_year_end"] = d.is_year_end.astype(int)
    out["days_to_month_end"] = (out[DATE_COL] + pd.offsets.MonthEnd(0) - out[DATE_COL]).dt.days.astype(int)
    out["days_since_start"] = (out[DATE_COL] - out[DATE_COL].min()).dt.days.astype(int)
    out["time_years"] = out["days_since_start"] / 365.25
    out["time_years_sq"] = out["time_years"] ** 2
    out["is_payday_window"] = ((d.day <= 5) | (d.day >= 25) | d.day.between(14, 17)).astype(int)
    out["is_double_day"] = (out["day_of_month"] == out["month"]).astype(int)
    out["is_mid_month"] = d.day.between(14, 16).astype(int)
    out["is_q4"] = d.month.isin([10, 11, 12]).astype(int)
    out["is_summer"] = d.month.isin([5, 6, 7, 8]).astype(int)

    # Both v3-style simple cyclical terms and v4-style multi-harmonic Fourier terms.
    for name, base_col, period, harmonics in [
        ("dow", "day_of_week", 7.0, 3),
        ("dom", "day_of_month", 31.0, 3),
        ("woy", "week_of_year", 52.18, 2),
        ("moy", "month", 12.0, 3),
        ("doy", "day_of_year", 365.25, 6),
    ]:
        x = out[base_col].astype(float)
        for k in range(1, harmonics + 1):
            out[f"{name}_sin_{k}"] = np.sin(2 * np.pi * k * x / period)
            out[f"{name}_cos_{k}"] = np.cos(2 * np.pi * k * x / period)
    return out


def add_tet_event_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dates = out[DATE_COL].values.astype("datetime64[D]")
    tet = TET_DATES.values.astype("datetime64[D]")
    nxt = np.clip(np.searchsorted(tet, dates, side="left"), 0, len(tet) - 1)
    prv = np.clip(nxt - 1, 0, len(tet) - 1)
    days_to = (tet[nxt] - dates).astype(int)
    days_since = (dates - tet[prv]).astype(int)
    nearest = np.minimum(np.abs(days_to), np.abs(days_since))
    out["days_to_tet"] = days_to
    out["days_since_tet"] = days_since
    out["abs_days_to_tet"] = nearest
    for scale in [3.0, 7.0, 14.0, 30.0]:
        out[f"tet_proximity_{int(scale)}"] = np.exp(-nearest / scale)
    out["is_tet_day"] = (days_to == 0).astype(int)
    out["is_tet_holiday"] = ((days_to == 0) | ((days_since >= 0) & (days_since <= 6))).astype(int)
    out["is_pre_tet_7d"] = ((days_to >= 1) & (days_to <= 7)).astype(int)
    out["is_pre_tet_14d"] = ((days_to >= 1) & (days_to <= 14)).astype(int)
    out["is_pre_tet_21d"] = ((days_to >= 1) & (days_to <= 21)).astype(int)
    out["is_pre_tet_45d"] = ((days_to >= 1) & (days_to <= 45)).astype(int)
    out["is_post_tet_7d"] = ((days_since >= 1) & (days_since <= 7)).astype(int)
    out["is_post_tet_14d"] = ((days_since >= 1) & (days_since <= 14)).astype(int)
    out["is_post_tet_30d"] = ((days_since >= 1) & (days_since <= 30)).astype(int)

    years = range(int(out["year"].min()), int(out["year"].max()) + 1)
    special_dates: Dict[str, List[pd.Timestamp]] = {k: [] for k in list(MEGA_DAYS) + ["black_friday", "cyber_monday"]}
    for y in years:
        for name, (m, dd) in MEGA_DAYS.items():
            dt = pd.Timestamp(year=y, month=m, day=dd)
            out[f"is_{name}"] = ((out[DATE_COL].dt.month == m) & (out[DATE_COL].dt.day == dd)).astype(int)
            special_dates[name].append(dt)
        bf = black_friday(y)
        special_dates["black_friday"].append(bf)
        special_dates["cyber_monday"].append(bf + pd.Timedelta(days=3))
    out["is_black_friday"] = out[DATE_COL].isin(pd.to_datetime(special_dates["black_friday"])).astype(int)
    out["is_cyber_monday"] = out[DATE_COL].isin(pd.to_datetime(special_dates["cyber_monday"])).astype(int)

    event_cols = [f"is_{k}" for k in MEGA_DAYS] + ["is_black_friday", "is_cyber_monday"]
    out["is_mega_event"] = out[event_cols].max(axis=1).astype(int)
    for name in ["dd_9_9", "dd_10_10", "dd_11_11", "dd_12_12", "black_friday", "cyber_monday"]:
        ev = pd.to_datetime(special_dates[name]).values.astype("datetime64[D]")
        idx = np.clip(np.searchsorted(ev, dates, side="left"), 0, len(ev) - 1)
        prev = np.clip(idx - 1, 0, len(ev) - 1)
        dt_next = (ev[idx] - dates).astype(int)
        dt_prev = (dates - ev[prev]).astype(int)
        near = np.minimum(np.abs(dt_next), np.abs(dt_prev))
        out[f"{name}_proximity"] = np.exp(-near / 7.0)
        out[f"is_pre_{name}_7d"] = ((dt_next >= 1) & (dt_next <= 7)).astype(int)
        out[f"is_post_{name}_7d"] = ((dt_prev >= 1) & (dt_prev <= 7)).astype(int)
    return out


def strip_year_from_promo_name(name: str) -> str:
    return re.sub(r"\s+\d{4}\s*$", "", str(name)).strip()


def expand_promotions(promos: Optional[pd.DataFrame], start: pd.Timestamp, end: pd.Timestamp, extend_recurring: bool) -> pd.DataFrame:
    cal = pd.DataFrame({DATE_COL: pd.date_range(start, end, freq="D")})
    cols = [
        "is_promo", "n_active_promos", "max_disc", "mean_discount", "n_promo_channels",
        "n_promo_categories", "n_stackable_promos", "has_fixed_promo", "has_percentage_promo",
        "synthetic_promo", "promo_intensity", "min_order_value_mean",
    ]
    if promos is None or promos.empty or "start_date" not in promos.columns:
        for c in cols:
            cal[c] = 0.0
        return cal

    p = promos.copy()
    if "end_date" not in p.columns:
        p["end_date"] = p["start_date"]
    p["start_date"] = pd.to_datetime(p["start_date"], errors="coerce")
    p["end_date"] = pd.to_datetime(p["end_date"], errors="coerce")
    p = p.dropna(subset=["start_date", "end_date"])
    p["discount_value"] = pd.to_numeric(p.get("discount_value", 0.0), errors="coerce").fillna(0.0)
    p["min_order_value"] = pd.to_numeric(p.get("min_order_value", 0.0), errors="coerce").fillna(0.0)
    if "promo_name" not in p.columns:
        p["promo_name"] = p.get("promo_id", "promo")
    p["promo_key"] = p["promo_name"].map(strip_year_from_promo_name)

    rows = []

    def append_range(row: dict, s, e, synthetic: int):
        s = max(pd.Timestamp(s), pd.Timestamp(start))
        e = min(pd.Timestamp(e), pd.Timestamp(end))
        if e < s:
            return
        promo_type = str(row.get("promo_type", "")).lower()
        channel = str(row.get("promo_channel", "unknown"))
        category = str(row.get("applicable_category", "all"))
        for dt in pd.date_range(s, e, freq="D"):
            rows.append({
                DATE_COL: dt,
                "promo_id": row.get("promo_id", row.get("promo_name", "promo")),
                "discount_value": float(row.get("discount_value", 0.0) or 0.0),
                "pct_discount": float(row.get("discount_value", 0.0) or 0.0) if promo_type == "percentage" else 0.0,
                "channel": channel,
                "category": category,
                "stackable": int(row.get("stackable_flag", 0) or 0),
                "has_fixed_promo": int(promo_type == "fixed"),
                "has_percentage_promo": int(promo_type == "percentage"),
                "synthetic_promo": int(synthetic),
                "min_order_value": float(row.get("min_order_value", 0.0) or 0.0),
            })

    for _, row in p.iterrows():
        rd = row.to_dict()
        s, e = rd["start_date"], rd["end_date"]
        if e < s:
            s, e = e, s
        append_range(rd, s, e, 0)

    if extend_recurring:
        min_year, max_year = int(pd.Timestamp(start).year), int(pd.Timestamp(end).year)
        for key, g in p.groupby("promo_key"):
            years = sorted(g["start_date"].dt.year.unique().tolist())
            if not years:
                continue
            cadence = 1 if len(years) < 2 else int(round(np.median(np.diff(years))))
            cadence = max(1, min(cadence, 5))
            template = g.sort_values("start_date").iloc[-1].to_dict()
            last_year = int(pd.Timestamp(template["start_date"]).year)
            duration = int((pd.Timestamp(template["end_date"]) - pd.Timestamp(template["start_date"])).days)
            month_day = (int(pd.Timestamp(template["start_date"]).month), int(pd.Timestamp(template["start_date"]).day))
            for y in range(min_year, max_year + 1):
                if y <= last_year or (y - last_year) % cadence != 0:
                    continue
                try:
                    s = pd.Timestamp(year=y, month=month_day[0], day=month_day[1])
                except ValueError:
                    continue
                append_range(template, s, s + pd.Timedelta(days=duration), 1)

    if not rows:
        for c in cols:
            cal[c] = 0.0
        return cal

    exp = pd.DataFrame(rows)
    agg = exp.groupby(DATE_COL, as_index=False).agg(
        n_active_promos=("promo_id", "nunique"),
        max_disc=("pct_discount", "max"),
        mean_discount=("discount_value", "mean"),
        n_promo_channels=("channel", "nunique"),
        n_promo_categories=("category", "nunique"),
        n_stackable_promos=("stackable", "sum"),
        has_fixed_promo=("has_fixed_promo", "max"),
        has_percentage_promo=("has_percentage_promo", "max"),
        synthetic_promo=("synthetic_promo", "max"),
        min_order_value_mean=("min_order_value", "mean"),
    )
    cal = cal.merge(agg, on=DATE_COL, how="left")
    fill_cols = [c for c in cal.columns if c != DATE_COL]
    cal[fill_cols] = cal[fill_cols].fillna(0.0)
    cal["is_promo"] = (cal["n_active_promos"] > 0).astype(int)
    cal["promo_intensity"] = cal["n_active_promos"] * (1.0 + cal["max_disc"] / 100.0)
    return cal


def build_feature_frame(sales: pd.DataFrame, sample: pd.DataFrame, promos: Optional[pd.DataFrame], cfg: V5Config) -> pd.DataFrame:
    start = min(sales["Date"].min(), sample["Date"].min())
    end = max(sales["Date"].max(), sample["Date"].max())
    df = pd.DataFrame({DATE_COL: pd.date_range(start, end, freq="D")})
    df = add_calendar_features(df)
    df = add_tet_event_features(df)
    promo_daily = expand_promotions(promos, start, end, cfg.use_extended_recurring_promos)
    df = df.merge(promo_daily, on=DATE_COL, how="left")
    for c in [c for c in promo_daily.columns if c != DATE_COL]:
        df[c] = df[c].fillna(0.0)

    for w in [7, 14, 30, 60]:
        df[f"promo_days_last_{w}"] = df["is_promo"].shift(1).rolling(w, min_periods=1).sum().fillna(0.0)
        df[f"promo_days_next_{w}"] = df["is_promo"][::-1].shift(1).rolling(w, min_periods=1).sum()[::-1].fillna(0.0)
    for event_col in ["is_pre_tet_21d", "is_tet_holiday", "is_mega_event", "is_weekend", "is_payday_window", "is_dd_11_11", "is_dd_12_12", "is_black_friday"]:
        if event_col in df.columns:
            df[f"promo_x_{event_col}"] = df["is_promo"] * df[event_col]
    return df.set_index(DATE_COL).sort_index()


def make_lag_features(series: pd.Series) -> pd.DataFrame:
    s = series.astype(float).copy()
    out = pd.DataFrame(index=s.index)
    for lag in LAGS:
        out[f"lag_{lag}"] = s.shift(lag)
    base = s.shift(1)
    for w in ROLL_WINDOWS:
        r = base.rolling(w, min_periods=max(3, w // 3))
        out[f"rmean_{w}"] = r.mean()
        out[f"rstd_{w}"] = r.std()
        out[f"rmin_{w}"] = r.min()
        out[f"rmax_{w}"] = r.max()
        out[f"rmed_{w}"] = r.median()
    out["diff_1"] = s.shift(1) - s.shift(2)
    out["diff_7"] = s.shift(1) - s.shift(8)
    out["diff_28"] = s.shift(1) - s.shift(29)
    out["yoy_ratio_365"] = s.shift(1) / (s.shift(365) + 1e-6)
    out["yoy_diff_365"] = s.shift(1) - s.shift(365)
    return out


def lag_row_from_history(history: pd.Series) -> pd.DataFrame:
    h = history.astype(float).dropna().sort_index()
    arr = h.to_numpy(dtype=float)
    n = len(arr)
    row: Dict[str, float] = {}
    for lag in LAGS:
        row[f"lag_{lag}"] = float(arr[-lag]) if n >= lag else np.nan
    for w in ROLL_WINDOWS:
        vals = arr[-w:] if n > 0 else np.array([])
        if len(vals) >= max(3, w // 3):
            row[f"rmean_{w}"] = float(np.nanmean(vals))
            row[f"rstd_{w}"] = float(np.nanstd(vals))
            row[f"rmin_{w}"] = float(np.nanmin(vals))
            row[f"rmax_{w}"] = float(np.nanmax(vals))
            row[f"rmed_{w}"] = float(np.nanmedian(vals))
        else:
            for stat in ["rmean", "rstd", "rmin", "rmax", "rmed"]:
                row[f"{stat}_{w}"] = np.nan
    row["diff_1"] = float(arr[-1] - arr[-2]) if n >= 2 else np.nan
    row["diff_7"] = float(arr[-1] - arr[-8]) if n >= 8 else np.nan
    row["diff_28"] = float(arr[-1] - arr[-29]) if n >= 29 else np.nan
    row["yoy_ratio_365"] = float(arr[-1] / (arr[-365] + 1e-6)) if n >= 365 else np.nan
    row["yoy_diff_365"] = float(arr[-1] - arr[-365]) if n >= 365 else np.nan
    return pd.DataFrame([row])


def fit_seasonal_profile(y: pd.Series) -> dict:
    y = y.astype(float).sort_index()
    if len(y) < 365:
        return {"level": float(y.median()), "profile": {}, "growth": 0.0, "last_date": y.index.max()}
    level = float(y.tail(min(365, len(y))).median())
    smooth = y.rolling(365, min_periods=120, center=True).median().ffill().bfill()
    ratio = (y / (smooth + 1e-6)).replace([np.inf, -np.inf], np.nan).clip(0.25, 4.0)
    keys = pd.Index([f"{d.month:02d}-{d.day:02d}" for d in y.index])
    prof = ratio.groupby(keys).median().to_dict()
    med_prof = np.nanmedian(list(prof.values())) if prof else 1.0
    prof = {k: float(v / med_prof) for k, v in prof.items() if np.isfinite(v)}
    if len(y) >= 730:
        recent = float(y.tail(365).mean())
        prev = float(y.iloc[-730:-365].mean())
        growth = recent / max(prev, 1e-6) - 1.0
    else:
        growth = 0.0
    growth = float(np.clip(growth, -0.30, 0.20))
    return {"level": level, "profile": prof, "growth": growth, "last_date": y.index.max()}


def seasonal_forecast(fit: dict, dates: pd.DatetimeIndex) -> np.ndarray:
    level = float(fit.get("level", 0.0))
    prof = fit.get("profile", {})
    growth = float(fit.get("growth", 0.0))
    last = pd.Timestamp(fit.get("last_date"))
    vals = []
    for d in pd.DatetimeIndex(dates):
        key = f"{d.month:02d}-{d.day:02d}"
        factor = float(prof.get(key, 1.0))
        years_ahead = max(0.0, (d - last).days / 365.25)
        vals.append(max(0.0, level * ((1.0 + growth) ** years_ahead) * factor))
    return np.asarray(vals, dtype=float)


def make_splits(n_rows: int, cfg: V5Config) -> List[Tuple[np.ndarray, np.ndarray]]:
    val_days = min(cfg.val_days, max(60, n_rows // 5))
    max_folds = cfg.max_folds
    start = n_rows - val_days - cfg.step_days * (max_folds - 1)
    while start < max(730, val_days) and max_folds > 1:
        max_folds -= 1
        start = n_rows - val_days - cfg.step_days * (max_folds - 1)
    if start < max(365, val_days):
        start = max(365, n_rows - val_days)
    folds = []
    train_end = start
    while train_end + val_days <= n_rows and len(folds) < max_folds:
        folds.append((np.arange(0, train_end), np.arange(train_end, train_end + val_days)))
        train_end += cfg.step_days
    return folds


def metrics(y_true, y_pred) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }


def xgb_params(candidate: dict, seed: int, cfg: V5Config) -> dict:
    keys = [
        "n_estimators", "learning_rate", "max_depth", "min_child_weight", "subsample",
        "colsample_bytree", "reg_alpha", "reg_lambda", "objective", "eval_metric",
    ]
    params = {k: candidate[k] for k in keys if k in candidate}
    params.update({
        "tree_method": "hist",
        "device": XGB_DEVICE,
        "random_state": seed,
        "n_jobs": cfg.n_jobs,
    })
    if candidate.get("objective") == "reg:tweedie":
        params["tweedie_variance_power"] = candidate.get("tweedie_variance_power", 1.15)
    return params


MODEL_LIBRARY = [
    {
        "name": "v3_log_d4_all", "kind": "det", "transform": "log1p", "recent_years": None,
        "objective": "reg:squarederror", "eval_metric": "mae", "n_estimators": 1400,
        "learning_rate": 0.045, "max_depth": 4, "min_child_weight": 6,
        "subsample": 0.90, "colsample_bytree": 0.90, "reg_alpha": 0.0, "reg_lambda": 1.0,
    },
    {
        "name": "v3_log_d6_all", "kind": "det", "transform": "log1p", "recent_years": None,
        "objective": "reg:squarederror", "eval_metric": "mae", "n_estimators": 1800,
        "learning_rate": 0.035, "max_depth": 6, "min_child_weight": 8,
        "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 0.01, "reg_lambda": 1.5,
    },
    {
        "name": "v4_log_d4_recent4", "kind": "det", "transform": "log1p", "recent_years": 4,
        "objective": "reg:squarederror", "eval_metric": "mae", "n_estimators": 1200,
        "learning_rate": 0.04, "max_depth": 4, "min_child_weight": 8,
        "subsample": 0.90, "colsample_bytree": 0.90, "reg_alpha": 0.01, "reg_lambda": 1.3,
    },
    {
        "name": "mae_raw_d4_recent4", "kind": "det", "transform": "none", "recent_years": 4,
        "objective": "reg:absoluteerror", "eval_metric": "mae", "n_estimators": 1000,
        "learning_rate": 0.045, "max_depth": 4, "min_child_weight": 8,
        "subsample": 0.90, "colsample_bytree": 0.90, "reg_alpha": 0.01, "reg_lambda": 1.0,
    },
    {
        "name": "tweedie_raw_d4_all", "kind": "det", "transform": "none", "recent_years": None,
        "objective": "reg:tweedie", "eval_metric": "mae", "tweedie_variance_power": 1.15,
        "n_estimators": 1200, "learning_rate": 0.04, "max_depth": 4, "min_child_weight": 8,
        "subsample": 0.88, "colsample_bytree": 0.88, "reg_alpha": 0.01, "reg_lambda": 1.0,
    },
    {
        "name": "lag_log_d3_recent4", "kind": "lag", "transform": "log1p", "recent_years": 4,
        "objective": "reg:squarederror", "eval_metric": "mae", "n_estimators": 700,
        "learning_rate": 0.045, "max_depth": 3, "min_child_weight": 10,
        "subsample": 0.90, "colsample_bytree": 0.90, "reg_alpha": 0.02, "reg_lambda": 2.0,
    },
]

RATIO_LIBRARY = [
    {
        "name": "ratio_xgb_d3_all", "kind": "det", "transform": "none", "recent_years": None,
        "objective": "reg:squarederror", "eval_metric": "mae", "n_estimators": 900,
        "learning_rate": 0.04, "max_depth": 3, "min_child_weight": 8,
        "subsample": 0.90, "colsample_bytree": 0.90, "reg_alpha": 0.01, "reg_lambda": 2.0,
    },
    {
        "name": "ratio_l1_d3_recent4", "kind": "det", "transform": "none", "recent_years": 4,
        "objective": "reg:absoluteerror", "eval_metric": "mae", "n_estimators": 700,
        "learning_rate": 0.04, "max_depth": 3, "min_child_weight": 8,
        "subsample": 0.90, "colsample_bytree": 0.90, "reg_alpha": 0.01, "reg_lambda": 2.0,
    },
]


def select_training_rows(train_dates: pd.DatetimeIndex, candidate: dict) -> pd.DatetimeIndex:
    years = candidate.get("recent_years", None)
    if years is None:
        return train_dates
    cutoff = train_dates.max() - pd.DateOffset(years=int(years))
    rows = train_dates[train_dates >= cutoff]
    if len(rows) < 700:
        rows = train_dates[-min(len(train_dates), 700):]
    return rows


def robust_sample_weight(y: pd.Series, dates: pd.DatetimeIndex, train_end: pd.Timestamp) -> np.ndarray:
    yv = y.astype(float).to_numpy()
    med = np.nanmedian(yv)
    mad = np.nanmedian(np.abs(yv - med)) + 1e-9
    z = np.abs(yv - med) / (1.4826 * mad)
    w = np.ones(len(yv), dtype=float)
    w[z > 5.0] *= 0.35
    # Recency weight: mild, not too aggressive.
    age_days = (pd.Timestamp(train_end) - pd.DatetimeIndex(dates)).days.astype(float)
    rec = np.exp(-np.maximum(age_days, 0) / (365.25 * 5.0))
    w *= 0.60 + 0.40 * rec
    return w


def fit_model_predict(X_train: pd.DataFrame, y_train: np.ndarray, X_pred: pd.DataFrame, candidate: dict, seed: int, cfg: V5Config, clip_min: float = 0.0, clip_max: Optional[float] = None):
    med = X_train.median(numeric_only=True)
    Xtr = X_train.replace([np.inf, -np.inf], np.nan).fillna(med).astype(np.float32)
    Xpr = X_pred.replace([np.inf, -np.inf], np.nan).fillna(med).astype(np.float32)
    transform = candidate.get("transform", "none")
    yf = np.log1p(np.clip(y_train, 0, None)) if transform == "log1p" else y_train
    params = xgb_params(candidate, seed, cfg)
    model = XGBRegressor(**params)
    try:
        model.fit(Xtr, yf, verbose=False)
    except Exception as exc:
        if params.get("objective") == "reg:absoluteerror":
            params["objective"] = "reg:squarederror"
            model = XGBRegressor(**params)
            model.fit(Xtr, yf, verbose=False)
            print(f"  [fallback] {candidate['name']} absoluteerror -> squarederror ({type(exc).__name__})")
        else:
            raise
    pred = model.predict(Xpr)
    if transform == "log1p":
        train_log_pred = model.predict(Xtr)
        resid = np.log1p(np.clip(y_train, 0, None)) - train_log_pred
        smear = float(np.mean(np.exp(np.clip(resid, -6.0, 6.0))))
        smear = float(np.clip(smear, 0.70, 1.50))
        pred = np.exp(pred) * smear - 1.0
    pred = np.asarray(pred, dtype=float)
    if clip_max is None:
        pred = np.clip(pred, clip_min, None)
    else:
        pred = np.clip(pred, clip_min, clip_max)
    return pred, model, med


def fit_model_object(X_train: pd.DataFrame, y_train: np.ndarray, candidate: dict, seed: int, cfg: V5Config):
    med = X_train.median(numeric_only=True)
    Xtr = X_train.replace([np.inf, -np.inf], np.nan).fillna(med).astype(np.float32)
    transform = candidate.get("transform", "none")
    yf = np.log1p(np.clip(y_train, 0, None)) if transform == "log1p" else y_train
    params = xgb_params(candidate, seed, cfg)
    model = XGBRegressor(**params)
    try:
        model.fit(Xtr, yf, verbose=False)
    except Exception as exc:
        if params.get("objective") == "reg:absoluteerror":
            params["objective"] = "reg:squarederror"
            model = XGBRegressor(**params)
            model.fit(Xtr, yf, verbose=False)
        else:
            raise
    return model, med


def predict_xgb_object(model, med: pd.Series, X_pred: pd.DataFrame, candidate: dict, clip_min: float, clip_max: Optional[float]) -> np.ndarray:
    Xpr = X_pred.replace([np.inf, -np.inf], np.nan).fillna(med).astype(np.float32)
    pred = model.predict(Xpr)
    if candidate.get("transform") == "log1p":
        pred = np.expm1(pred)
    pred = np.asarray(pred, dtype=float)
    if clip_max is None:
        return np.clip(pred, clip_min, None)
    return np.clip(pred, clip_min, clip_max)


def fit_blend_weights(y_true, pred_matrix, n_samples: int, seed: int, metric: str = "mae") -> Tuple[np.ndarray, float]:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(pred_matrix, dtype=float)
    k = p.shape[1]
    rng = np.random.default_rng(seed)
    best_w = np.zeros(k, dtype=float)
    best_score = float("inf")
    for j in range(k):
        pred = p[:, j]
        score = mean_absolute_error(y, pred) if metric == "mae" else math.sqrt(mean_squared_error(y, pred))
        if score < best_score:
            best_score = float(score)
            best_w[:] = 0
            best_w[j] = 1.0
    for _ in range(n_samples):
        # sparse-ish Dirichlet sometimes produces stronger single/double model blends.
        alpha = rng.choice([0.30, 0.60, 1.00, 1.50])
        w = rng.dirichlet(np.ones(k) * alpha)
        pred = p @ w
        score = mean_absolute_error(y, pred) if metric == "mae" else math.sqrt(mean_squared_error(y, pred))
        if score < best_score:
            best_score = float(score)
            best_w = w
    return best_w, best_score


def fit_mae_calibration(y_true, pred, clip_min: float = 0.0, clip_max: Optional[float] = None) -> Tuple[float, float, float]:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(pred, dtype=float)
    best = (1.0, 0.0, mean_absolute_error(y, p))
    for mult in np.linspace(0.82, 1.18, 145):
        bias = float(np.median(y - p * mult))
        pp = p * mult + bias
        if clip_max is None:
            pp = np.clip(pp, clip_min, None)
        else:
            pp = np.clip(pp, clip_min, clip_max)
        score = float(mean_absolute_error(y, pp))
        if score < best[2]:
            best = (float(mult), bias, score)
    return best


def horizon_bucket(h: int) -> str:
    if h <= 30:
        return "h001_030"
    if h <= 90:
        return "h031_090"
    if h <= 180:
        return "h091_180"
    if h <= 365:
        return "h181_365"
    return "h366_plus"


def fit_bucket_weights(oof: pd.DataFrame, candidate_cols: List[str], cfg: V5Config) -> Dict[str, np.ndarray]:
    weights = {}
    for b, g in oof.dropna(subset=candidate_cols + ["actual", "horizon_bucket"]).groupby("horizon_bucket"):
        if len(g) < max(80, len(candidate_cols) * 12):
            continue
        w, _ = fit_blend_weights(g["actual"].values, g[candidate_cols].values, max(2000, cfg.n_blend_samples // 3), cfg.seed + len(b))
        weights[b] = w
    return weights


def apply_bucket_weights(pred_df: pd.DataFrame, candidate_cols: List[str], global_w: np.ndarray, bucket_w: Dict[str, np.ndarray]) -> np.ndarray:
    mat = pred_df[candidate_cols].to_numpy(dtype=float)
    out = mat @ global_w
    if "horizon_bucket" in pred_df.columns:
        for b, w in bucket_w.items():
            mask = pred_df["horizon_bucket"].values == b
            if mask.any():
                out[mask] = mat[mask] @ w
    return out


class DampedKalmanResidual:
    def __init__(self, q: float = 0.01, r: float = 1.0, damping: float = 0.98):
        self.F = np.array([[1.0, 1.0], [0.0, damping]], dtype=float)
        self.H = np.array([[1.0, 0.0]], dtype=float)
        self.Q = np.diag([q, q * 0.05])
        self.R = float(r)
        self.x = np.zeros((2, 1), dtype=float)
        self.P = np.eye(2) * 10.0

    def initialize(self, residuals: Sequence[float]):
        r = np.asarray(list(residuals), dtype=float)
        r = r[np.isfinite(r)]
        if len(r) == 0:
            return
        tail = r[-min(180, len(r)):]
        scale = np.median(np.abs(tail - np.median(tail))) * 1.4826 + 1e-6
        self.x[0, 0] = float(np.median(tail))
        self.x[1, 0] = float(np.median(np.diff(tail))) if len(tail) > 8 else 0.0
        self.P = np.eye(2) * max(scale ** 2, 1.0)
        self.R = max(scale ** 2, 1.0)
        self.Q = np.diag([max((scale * 0.05) ** 2, 1e-6), max((scale * 0.005) ** 2, 1e-8)])

    def predict(self) -> float:
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return float(self.H @ self.x)

    def project(self, n: int) -> np.ndarray:
        return np.asarray([self.predict() for _ in range(n)], dtype=float)


def kalman_project(base_pred: np.ndarray, residuals_tail: Sequence[float], strength: float, clip_min: float = 0.0, clip_max: Optional[float] = None) -> np.ndarray:
    kf = DampedKalmanResidual()
    kf.initialize(residuals_tail)
    corr = kf.project(len(base_pred))
    pred = np.asarray(base_pred, dtype=float) + float(strength) * corr
    if clip_max is None:
        return np.clip(pred, clip_min, None)
    return np.clip(pred, clip_min, clip_max)


def print_metric_table(title: str, rows: Dict[str, Dict[str, float]]):
    print("\n" + title)
    print("-" * len(title))
    for k, m in rows.items():
        print(f"{k:24s} MAE={m['MAE']:,.2f} RMSE={m['RMSE']:,.2f} R2={m['R2']:.4f}")


def train_target_hybrid(target_name: str, y: pd.Series, X_det_all: pd.DataFrame, dates: pd.DatetimeIndex, test_dates: pd.DatetimeIndex, cfg: V5Config, library: List[dict], clip_min: float = 0.0, clip_max: Optional[float] = None) -> dict:
    n = len(dates)
    folds = make_splits(n, cfg)
    print(f"\n[{target_name}] {len(folds)} folds, {len(X_det_all.columns)} deterministic features, device={XGB_DEVICE}")
    for i, (tr, va) in enumerate(folds, 1):
        print(f"  fold {i}: train={len(tr)} valid={len(va)} {dates[va[0]].date()} -> {dates[va[-1]].date()}")

    y_values = y.loc[dates].to_numpy(dtype=float)
    oof = pd.DataFrame({"date": dates, "actual": y_values})
    oof["fold"] = np.nan
    oof["horizon"] = np.nan
    oof["horizon_bucket"] = None
    candidate_cols = ["seasonal_anchor"] + [c["name"] for c in library if (c.get("kind") != "lag" or cfg.use_recursive_lag_candidate)]
    for c in candidate_cols:
        oof[c] = np.nan

    for fold_id, (tr_idx, va_idx) in enumerate(folds):
        tr_dates = dates[tr_idx]
        va_dates = dates[va_idx]
        y_tr = y.loc[tr_dates]
        seasonal_fit = fit_seasonal_profile(y_tr)
        oof.loc[va_idx, "seasonal_anchor"] = seasonal_forecast(seasonal_fit, va_dates)
        oof.loc[va_idx, "fold"] = fold_id
        oof.loc[va_idx, "horizon"] = np.arange(1, len(va_idx) + 1)
        oof.loc[va_idx, "horizon_bucket"] = [horizon_bucket(h) for h in range(1, len(va_idx) + 1)]

        lag_all_train = None
        for cand in library:
            if cand.get("kind") == "lag" and not cfg.use_recursive_lag_candidate:
                continue
            sample_dates = select_training_rows(tr_dates, cand)
            if cand.get("kind") == "lag":
                if lag_all_train is None:
                    lag_all_train = pd.concat([X_det_all, make_lag_features(y.loc[dates])], axis=1)
                train_cols = list(lag_all_train.columns)
                X_tr = lag_all_train.loc[sample_dates, train_cols]
                y_sample = y.loc[sample_dates].to_numpy(dtype=float)
                model, med = fit_model_object(X_tr, y_sample, cand, cfg.seed + fold_id, cfg)
                hist = y.loc[tr_dates].astype(float).copy()
                preds = []
                for d in va_dates:
                    row_lag = lag_row_from_history(hist)
                    row_lag.index = [d]
                    row = pd.concat([X_det_all.loc[[d]].copy(), row_lag], axis=1).reindex(columns=train_cols)
                    p = predict_xgb_object(model, med, row, cand, clip_min, clip_max)[0]
                    preds.append(p)
                    hist.loc[d] = p
                oof.loc[va_idx, cand["name"]] = preds
            else:
                sample_dates = select_training_rows(tr_dates, cand)
                X_tr = X_det_all.loc[sample_dates]
                X_va = X_det_all.loc[va_dates]
                y_sample = y.loc[sample_dates].to_numpy(dtype=float)
                pred_va, _, _ = fit_model_predict(X_tr, y_sample, X_va, cand, cfg.seed + fold_id, cfg, clip_min, clip_max)
                oof.loc[va_idx, cand["name"]] = pred_va

    usable = oof.dropna(subset=candidate_cols + ["actual"]).copy()
    pred_mat = usable[candidate_cols].to_numpy(dtype=float)
    global_w, _ = fit_blend_weights(usable["actual"].values, pred_mat, cfg.n_blend_samples, cfg.seed)
    bucket_w = fit_bucket_weights(usable, candidate_cols, cfg) if cfg.use_horizon_bucket_blend else {}

    oof.loc[usable.index, "blend_global"] = pred_mat @ global_w
    # Use bucket weights where available.
    usable_pred = usable.copy()
    usable_pred["blend_bucket"] = apply_bucket_weights(usable_pred, candidate_cols, global_w, bucket_w)
    oof.loc[usable.index, "blend_bucket"] = usable_pred["blend_bucket"].values

    base_for_cal = oof.loc[usable.index, "blend_bucket"].values
    mult, bias, _ = fit_mae_calibration(usable["actual"].values, base_for_cal, clip_min, clip_max)
    oof.loc[usable.index, "calibrated"] = base_for_cal * mult + bias
    if clip_max is None:
        oof.loc[usable.index, "calibrated"] = np.clip(oof.loc[usable.index, "calibrated"], clip_min, None)
    else:
        oof.loc[usable.index, "calibrated"] = np.clip(oof.loc[usable.index, "calibrated"], clip_min, clip_max)

    # Kalman gate: only keep nonzero strength if it improves OOF MAE.
    kalman_strength = 0.0
    kalman_metrics = {}
    if cfg.use_kalman_gate:
        strengths = [0.0, 0.10, 0.20, 0.35, 0.50]
        best_score = mean_absolute_error(usable["actual"].values, oof.loc[usable.index, "calibrated"].values)
        best_pred = oof.loc[usable.index, "calibrated"].values
        for s in strengths:
            pred_all = np.full(len(oof), np.nan)
            for f, g in oof.dropna(subset=["fold", "calibrated"]).groupby("fold"):
                idx = g.index.to_numpy()
                # In true test we do not know residuals in the future block. Initialize from residuals
                # before the fold when available; if unavailable, use earlier OOF residuals.
                prior = oof.loc[oof.index < idx.min()].dropna(subset=["actual", "calibrated"])
                resid_tail = (prior["actual"] - prior["calibrated"]).tail(365).values
                pred_all[idx] = kalman_project(g["calibrated"].values, resid_tail, s, clip_min, clip_max)
            mask = np.isfinite(pred_all) & np.isfinite(oof["actual"].values)
            if mask.any():
                sc = mean_absolute_error(oof.loc[mask, "actual"].values, pred_all[mask])
                kalman_metrics[str(s)] = float(sc)
                if sc < best_score:
                    best_score = float(sc)
                    best_pred = pred_all[usable.index]
                    kalman_strength = float(s)
        oof.loc[usable.index, "kalman"] = best_pred
    else:
        oof.loc[usable.index, "kalman"] = oof.loc[usable.index, "calibrated"].values

    result_metrics = {
        "seasonal_anchor": metrics(usable["actual"], usable["seasonal_anchor"]),
        "blend_global": metrics(usable["actual"], oof.loc[usable.index, "blend_global"]),
        "blend_bucket": metrics(usable["actual"], oof.loc[usable.index, "blend_bucket"]),
        "calibrated": metrics(usable["actual"], oof.loc[usable.index, "calibrated"]),
        "kalman_gated": metrics(usable["actual"], oof.loc[usable.index, "kalman"]),
    }
    for c in candidate_cols:
        result_metrics[c] = metrics(usable["actual"], usable[c])
    print_metric_table(target_name, result_metrics)
    print("  global weights:", {candidate_cols[i]: round(float(global_w[i]), 5) for i in range(len(candidate_cols))})
    if bucket_w:
        print("  bucket weights:", {b: {candidate_cols[i]: round(float(w[i]), 4) for i in range(len(candidate_cols))} for b, w in bucket_w.items()})
    print("  calibration:", {"multiplier": mult, "bias": bias})
    print("  kalman_strength:", kalman_strength, "grid_mae:", kalman_metrics)

    # Final forecast: train full candidates and blend/calibrate using OOF-learned parameters.
    final_pred_df = pd.DataFrame(index=test_dates)
    final_pred_df["seasonal_anchor"] = seasonal_forecast(fit_seasonal_profile(y.loc[dates]), test_dates)
    final_pred_df["horizon"] = np.arange(1, len(test_dates) + 1)
    final_pred_df["horizon_bucket"] = [horizon_bucket(h) for h in final_pred_df["horizon"]]

    lag_all_full = None
    for cand in library:
        if cand.get("kind") == "lag" and not cfg.use_recursive_lag_candidate:
            continue
        sample_dates = select_training_rows(dates, cand)
        if cand.get("kind") == "lag":
            if lag_all_full is None:
                lag_all_full = pd.concat([X_det_all, make_lag_features(y.loc[dates])], axis=1)
            train_cols = list(lag_all_full.columns)
            X_tr = lag_all_full.loc[sample_dates, train_cols]
            y_sample = y.loc[sample_dates].to_numpy(dtype=float)
            model, med = fit_model_object(X_tr, y_sample, cand, cfg.seed + 999, cfg)
            hist = y.loc[dates].astype(float).copy()
            vals = []
            for d in test_dates:
                row_lag = lag_row_from_history(hist)
                row_lag.index = [d]
                row = pd.concat([X_det_all.loc[[d]].copy(), row_lag], axis=1).reindex(columns=train_cols)
                p = predict_xgb_object(model, med, row, cand, clip_min, clip_max)[0]
                vals.append(p)
                hist.loc[d] = p
            final_pred_df[cand["name"]] = vals
        else:
            X_tr = X_det_all.loc[sample_dates]
            X_te = X_det_all.loc[test_dates]
            y_sample = y.loc[sample_dates].to_numpy(dtype=float)
            pred_te, _, _ = fit_model_predict(X_tr, y_sample, X_te, cand, cfg.seed + 999, cfg, clip_min, clip_max)
            final_pred_df[cand["name"]] = pred_te

    final_base = apply_bucket_weights(final_pred_df, candidate_cols, global_w, bucket_w)
    final_cal = final_base * mult + bias
    if clip_max is None:
        final_cal = np.clip(final_cal, clip_min, None)
    else:
        final_cal = np.clip(final_cal, clip_min, clip_max)
    # Use last available OOF residuals for a safe residual projection; gated strength is often zero.
    resid_tail = (usable["actual"].values - oof.loc[usable.index, "calibrated"].values)[-365:]
    final_kalman = kalman_project(final_cal, resid_tail, kalman_strength, clip_min, clip_max)
    final_pred_df["blend_bucket"] = final_base
    final_pred_df["calibrated"] = final_cal
    final_pred_df["kalman_gated"] = final_kalman

    return {
        "target": target_name,
        "oof": oof,
        "final_components": final_pred_df.reset_index().rename(columns={"index": "Date"}),
        "test_pred": final_kalman,
        "candidate_cols": candidate_cols,
        "global_weights": {candidate_cols[i]: float(global_w[i]) for i in range(len(candidate_cols))},
        "bucket_weights": {b: {candidate_cols[i]: float(w[i]) for i in range(len(candidate_cols))} for b, w in bucket_w.items()},
        "calibration": {"multiplier": float(mult), "bias": float(bias)},
        "kalman_strength": kalman_strength,
        "kalman_grid_mae": kalman_metrics,
        "metrics": result_metrics,
    }


def locate_anchor_submission(cfg: V5Config, output_dir: Path) -> Optional[Path]:
    candidates = []
    if cfg.anchor_submission_path:
        candidates.append(Path(cfg.anchor_submission_path))
    candidates += [
        Path("/mnt/data/submission.csv"),
        Path.cwd() / "submission_best.csv",
        Path.cwd() / "best_submission.csv",
        Path.cwd() / "submission_anchor.csv",
        output_dir / "submission_best.csv",
    ]
    for p in candidates:
        if p.exists() and p.is_file():
            return p
    return None


def create_anchor_blends(model_sub: pd.DataFrame, sample: pd.DataFrame, cfg: V5Config, output_dir: Path) -> Optional[dict]:
    anchor_path = locate_anchor_submission(cfg, output_dir)
    if anchor_path is None:
        return None
    try:
        anchor = pd.read_csv(anchor_path)
        anchor["Date"] = pd.to_datetime(anchor["Date"], errors="coerce")
        anchor = anchor.merge(sample[["Date"]], on="Date", how="right")
        for c in ["Revenue", "COGS"]:
            anchor[c] = pd.to_numeric(anchor[c], errors="coerce")
        if anchor[["Revenue", "COGS"]].isna().any().any():
            print(f"[anchor] skipped because {anchor_path} has missing values after alignment")
            return None
    except Exception as e:
        print(f"[anchor] skipped: {e}")
        return None

    scaled = model_sub.copy()
    scale = {}
    for c in ["Revenue", "COGS"]:
        fac = float(anchor[c].mean() / max(model_sub[c].mean(), 1e-9))
        scale[c] = fac
        scaled[c] = model_sub[c] * fac

    blend_info = {"anchor_path": str(anchor_path), "scale_model_to_anchor": scale, "files": []}
    for w in cfg.anchor_blend_weights:
        blend = anchor.copy()
        for c in ["Revenue", "COGS"]:
            blend[c] = (1.0 - w) * anchor[c] + w * scaled[c]
            blend[c] = blend[c].clip(lower=0).round(2)
        blend["Date"] = blend["Date"].dt.strftime("%Y-%m-%d")
        fname = f"submission_v5_anchor{int((1-w)*100)}_model{int(w*100)}.csv"
        path = output_dir / fname
        blend[["Date", "Revenue", "COGS"]].to_csv(path, index=False)
        blend_info["files"].append(fname)

    rec_w = cfg.recommended_anchor_weight
    rec_name = f"submission_v5_anchor{int((1-rec_w)*100)}_model{int(rec_w*100)}.csv"
    rec_path = output_dir / rec_name
    if rec_path.exists():
        # Make a copy with a stable recommended name.
        rec = pd.read_csv(rec_path)
        rec.to_csv(output_dir / "submission_v5_recommended.csv", index=False)
        blend_info["recommended"] = "submission_v5_recommended.csv"
    return blend_info



def apply_fast_mode_for_local_run():
    """Reduce runtime when V5_FAST=1. Full notebook defaults remain stronger for Kaggle."""
    for lib in (MODEL_LIBRARY, RATIO_LIBRARY):
        for cand in lib:
            cand["n_estimators"] = min(int(cand.get("n_estimators", 500)), int(os.environ.get("V5_FAST_ESTIMATORS", "80")))
            cand["learning_rate"] = max(float(cand.get("learning_rate", 0.04)), 0.055)


def run_pipeline(cfg: Optional[V5Config] = None):
    cfg = cfg or CFG
    set_seed(cfg.seed)
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    data_dir, sales, sample, promos = load_data(cfg)
    print("DATA_DIR:", data_dir)
    print("sales:", sales.shape, sales["Date"].min().date(), "->", sales["Date"].max().date())
    print("sample:", sample.shape, sample["Date"].min().date(), "->", sample["Date"].max().date())
    print("promotions:", None if promos is None else promos.shape)

    feat = build_feature_frame(sales, sample, promos, cfg)
    feature_cols = [c for c in feat.columns if pd.api.types.is_numeric_dtype(feat[c])]
    feat = feat[feature_cols].replace([np.inf, -np.inf], np.nan)
    train_dates = pd.DatetimeIndex(sales["Date"])
    test_dates = pd.DatetimeIndex(sample["Date"])
    X_all = feat.reindex(pd.DatetimeIndex(list(train_dates) + list(test_dates)))
    print("n_features:", len(feature_cols))

    y_rev = pd.Series(sales["Revenue"].to_numpy(dtype=float), index=train_dates, name="Revenue")
    y_cogs = pd.Series(sales["COGS"].to_numpy(dtype=float), index=train_dates, name="COGS")
    y_ratio = (y_cogs / y_rev.clip(lower=1.0)).clip(0.01, 3.0)
    ratio_clip = (max(0.01, float(y_ratio.quantile(0.005)) * 0.9), min(3.0, float(y_ratio.quantile(0.995)) * 1.1))

    rev_res = train_target_hybrid("Revenue", y_rev, X_all, train_dates, test_dates, cfg, MODEL_LIBRARY, 0.0, None)
    cogs_direct_res = train_target_hybrid("COGS_direct", y_cogs, X_all, train_dates, test_dates, cfg, MODEL_LIBRARY, 0.0, None)
    ratio_res = train_target_hybrid("COGS_ratio", y_ratio, X_all, train_dates, test_dates, cfg, RATIO_LIBRARY, ratio_clip[0], ratio_clip[1])

    # Reconcile COGS: direct vs Revenue*ratio, optimize on OOF MAE.
    rev_oof = rev_res["oof"][["date", "kalman"]].rename(columns={"kalman": "rev_pred"})
    cd_oof = cogs_direct_res["oof"][["date", "actual", "kalman"]].rename(columns={"kalman": "cogs_direct_pred"})
    ratio_oof = ratio_res["oof"][["date", "kalman"]].rename(columns={"kalman": "ratio_pred"})
    recon = cd_oof.merge(rev_oof, on="date", how="left").merge(ratio_oof, on="date", how="left")
    recon["cogs_from_ratio"] = recon["rev_pred"] * recon["ratio_pred"]
    usable = recon.dropna(subset=["actual", "cogs_direct_pred", "cogs_from_ratio"]).copy()
    best_alpha = 1.0
    best_score = float(mean_absolute_error(usable["actual"], usable["cogs_direct_pred"]))
    for alpha in np.linspace(0.0, 1.0, 101):
        pred = alpha * usable["cogs_direct_pred"] + (1.0 - alpha) * usable["cogs_from_ratio"]
        score = float(mean_absolute_error(usable["actual"], pred))
        if score < best_score:
            best_score = score
            best_alpha = float(alpha)
    usable["cogs_reconciled"] = best_alpha * usable["cogs_direct_pred"] + (1.0 - best_alpha) * usable["cogs_from_ratio"]
    recon_metrics = {
        "direct": metrics(usable["actual"], usable["cogs_direct_pred"]),
        "from_ratio": metrics(usable["actual"], usable["cogs_from_ratio"]),
        "reconciled": metrics(usable["actual"], usable["cogs_reconciled"]),
    }
    print_metric_table("COGS reconciliation", recon_metrics)
    print("  best_alpha_direct:", best_alpha)

    revenue_final = np.asarray(rev_res["test_pred"], dtype=float)
    cogs_direct_final = np.asarray(cogs_direct_res["test_pred"], dtype=float)
    ratio_final = np.asarray(ratio_res["test_pred"], dtype=float)
    cogs_from_ratio_final = revenue_final * ratio_final
    cogs_final = best_alpha * cogs_direct_final + (1.0 - best_alpha) * cogs_from_ratio_final
    cogs_final = np.clip(cogs_final, 0.0, None)

    submission_model = pd.DataFrame({
        "Date": sample["Date"].dt.strftime("%Y-%m-%d"),
        "Revenue": np.round(revenue_final, 2),
        "COGS": np.round(cogs_final, 2),
    })
    model_path = output_dir / "submission_v5_hybrid_model_only.csv"
    submission_model.to_csv(model_path, index=False)

    blend_info = create_anchor_blends(submission_model.copy(), sample, cfg, output_dir)
    if blend_info is None:
        submission_model.to_csv(output_dir / "submission_v5_recommended.csv", index=False)

    # Save diagnostics.
    rev_res["oof"].to_csv(output_dir / "v5_oof_revenue.csv", index=False)
    cogs_direct_res["oof"].to_csv(output_dir / "v5_oof_cogs_direct.csv", index=False)
    ratio_res["oof"].to_csv(output_dir / "v5_oof_cogs_ratio.csv", index=False)
    rev_res["final_components"].to_csv(output_dir / "v5_final_revenue_components.csv", index=False)
    cogs_direct_res["final_components"].to_csv(output_dir / "v5_final_cogs_direct_components.csv", index=False)
    ratio_res["final_components"].to_csv(output_dir / "v5_final_cogs_ratio_components.csv", index=False)

    summary = {
        "config": asdict(cfg),
        "data_dir": str(data_dir),
        "xgb_device": XGB_DEVICE,
        "feature_count": len(feature_cols),
        "ratio_clip": ratio_clip,
        "revenue": {
            "metrics": rev_res["metrics"],
            "global_weights": rev_res["global_weights"],
            "bucket_weights": rev_res["bucket_weights"],
            "calibration": rev_res["calibration"],
            "kalman_strength": rev_res["kalman_strength"],
            "kalman_grid_mae": rev_res["kalman_grid_mae"],
        },
        "cogs_direct": {
            "metrics": cogs_direct_res["metrics"],
            "global_weights": cogs_direct_res["global_weights"],
            "bucket_weights": cogs_direct_res["bucket_weights"],
            "calibration": cogs_direct_res["calibration"],
            "kalman_strength": cogs_direct_res["kalman_strength"],
            "kalman_grid_mae": cogs_direct_res["kalman_grid_mae"],
        },
        "cogs_ratio": {
            "metrics": ratio_res["metrics"],
            "global_weights": ratio_res["global_weights"],
            "bucket_weights": ratio_res["bucket_weights"],
            "calibration": ratio_res["calibration"],
            "kalman_strength": ratio_res["kalman_strength"],
            "kalman_grid_mae": ratio_res["kalman_grid_mae"],
        },
        "cogs_reconciliation": {"alpha_direct": best_alpha, "metrics": recon_metrics},
        "anchor_blends": blend_info,
        "outputs": {
            "model_only": "submission_v5_hybrid_model_only.csv",
            "recommended": "submission_v5_recommended.csv",
        },
    }
    with open(output_dir / "v5_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\nSaved:", model_path)
    print("Saved:", output_dir / "submission_v5_recommended.csv")
    print("Saved:", output_dir / "v5_summary.json")
    return summary



## Time-Aware Cross-Validation

Cross-validation trong notebook l? walk-forward/expanding-window:

- `make_splits(n_rows, cfg)` ch?n c?c block validation ? cu?i chu?i, m?i block d?i `cfg.val_days`.
- V?i m?i fold trong `train_target_hybrid`, `tr_dates = dates[tr_idx]` lu?n n?m tr??c `va_dates = dates[va_idx]`.
- Seasonal profile, XGBoost models, blend inputs c?a t?ng fold ch? fit t? `tr_dates` ho?c t? candidate-specific recent subset b?n trong `tr_dates`.
- Nh?nh lag d? b?o validation theo ki?u recursive: b?t ??u b?ng l?ch s? train, sau ?? t?ng ng?y validation ch? th?m prediction c?a ng?y tr??c ??, kh?ng th?m actual validation t??ng lai.
- Metrics, blend weights, calibration v? Kalman gate ???c h?c t? OOF validation, sau ?? final model m?i train l?i tr?n full history ?? forecast sample period.

?i?m c?n tr?nh b?y khi b?o c?o: ??y kh?ng ph?i random CV; ??y l? pseudo out-of-sample backtest theo chi?u th?i gian, ph? h?p b?i to?n forecast d?i h?n.


## Run

Ch?y cell d??i ?? sinh c?c file ch?nh:

- `submission_v5_hybrid_model_only.csv`
- `submission_v5_recommended.csv`
- `v5_summary.json`
- `v5_oof_revenue.csv`, `v5_oof_cogs_direct.csv`, `v5_oof_cogs_ratio.csv`
- `v5_final_revenue_components.csv`, `v5_final_cogs_direct_components.csv`, `v5_final_cogs_ratio_components.csv`

G?i ?: ch?y full tr?n Kaggle. Khi th? nhanh local, ??t `RUN_FAST_LOCAL=True` ho?c `use_recursive_lag_candidate=False`.


In [2]:

# ============================ RUN V5 HYBRID ============================
# Full competition mode. On Kaggle this writes to /kaggle/working.
# If runtime is tight, set RUN_FAST_LOCAL=True or use_recursive_lag_candidate=False.

RUN_FAST_LOCAL = False
if RUN_FAST_LOCAL:
    apply_fast_mode_for_local_run()

cfg = V5Config(
    output_dir="/kaggle/working" if Path("/kaggle/working").exists() else "/mnt/data/v5_outputs",
    data_dir=None,  # or set to "/kaggle/input/..." explicitly
    anchor_submission_path=None,  # optional: path to your current best submission for conservative probe blends
    use_recursive_lag_candidate=True,
    use_horizon_bucket_blend=True,
    use_kalman_gate=True,
    n_jobs=2,
)

summary = run_pipeline(cfg)
summary


FileNotFoundError: Cannot find sales.csv and sample_submission.csv. Set cfg.data_dir explicitly.

## Explainability And Constraint Checklist

Ph?n explainability kh?ng thay ??i forecast. Notebook hi?n gi?i th?ch m? h?nh theo t?ng ??ng g?p, t??ng ???ng model attribution:

| Artifact | C?ch ??c |
|---|---|
| `v5_summary.json` | Xem `global_weights`, `bucket_weights`, `calibration`, `kalman_strength` cho t?ng target. ??y l? ??ng g?p c? th? c?a t?ng candidate model v? t?ng horizon bucket. |
| `v5_oof_*.csv` | So s?nh actual v?i t?ng candidate (`seasonal_anchor`, XGBoost variants), `blend_global`, `blend_bucket`, `calibrated`, `kalman`. |
| `v5_final_*_components.csv` | Xem t?ng component ??ng g?p th? n?o v?o forecast cu?i tr?n test horizon. |
| Console metric tables | MAE/RMSE/R2 c?a t?ng candidate v? c?c b??c blend/calibration/Kalman. |

N?u b?o c?o y?u c?u SHAP feature-level tuy?t ??i, n?n ch?y SHAP ? m?t b?n copy ri?ng ?? tr?nh ?nh h??ng runtime/k?t qu? notebook submit. B?n n?y gi? nguy?n logic c? v? d?ng c?c artifact tr?n l?m gi?i th?ch t??ng ???ng, t?p trung v?o m? h?nh n?o ???c tin nhi?u h?n, ? horizon n?o, v? post-processing n?o ???c gate gi? l?i.

Checklist r?ng bu?c:

- `sample` ch? gi? `Date` trong `load_data`, kh?ng d?ng target test.
- Feature t??ng lai l? calendar/event/promotion-plan, ??u l? known-future.
- Target lag/rolling ??u shift t? qu? kh? ho?c recursive prediction, kh?ng ??c actual validation/test t??ng lai trong v?ng d? b?o.
- `COGS_ratio = COGS / Revenue` ch? t?nh t? train history; test COGS ???c reconcile t? forecast Revenue v? forecast ratio.
- Anchor submission n?u c? ch? d?ng ?? t?o conservative probe blend sau khi model ?? forecast, kh?ng d?ng l?m nh?n train.
- Submission cu?i clip non-negative v? gi? ??ng c?t `Date`, `Revenue`, `COGS`.


## Submission Strategy

- N?u kh?ng truy?n `anchor_submission_path`, file recommended ch?nh l? model-only.
- N?u truy?n submission t?t nh?t hi?n t?i, notebook t?o probe blend r?t b?o th?: 95/5, 90/10, 85/15, 80/20, 70/30 sau khi scale model forecast v? c?ng mean v?i anchor.
- N?n submit probe 95/5 tr??c; n?u t?t h?n m?i t?ng weight model.

Ph?n anchor blend l? h?u x? l? submission, kh?ng tham gia train v? kh?ng l?m r? r? label.
